In [ ]:
# ==============================================================================
# [GLOBAL CELL 1] KONFIGURASI SENTRAL & SEEDING (MODEL EVALUATION & BENCHMARK)
# ==============================================================================
import os
import random
import glob
from enum import Enum
from typing import Tuple, List

import numpy as np
import tensorflow as tf

# ------------------------------------------------------------------------------
# 1. ENUMERASI ARSITEKTUR & TIPE MODEL
# ------------------------------------------------------------------------------
class ModelArch(str, Enum):
    MOBILENET_V3 = 'MobileNetV3Small'
    EFFICIENTNET_B0 = 'EfficientNetB0'

class ModelType(str, Enum):
    TRAINED = 'trained_baseline'
    PRUNED = 'pruned_recovered'
    QUANTIZED = 'quantized_int8'

# ------------------------------------------------------------------------------
# 2. SENTRALISASI KONFIGURASI (Standar 2 & 4)
# ------------------------------------------------------------------------------
class EvalConfig:
    """Konfigurasi utama untuk Fase Evaluasi dan Benchmarking."""
    
    PATH_TEST_FFB: str = "/kaggle/input/datasets/marrioqqqq/dataset-final/test"
    IMG_SIZE: Tuple[int, int] = (224, 224)
    BATCH_SIZE: int = 32
    NUM_CLASSES: int = 3
    
    # --------------------------------------------------------------------------
    # PATH RESOLVER (Sistem Pencarian Model Cerdas)
    # --------------------------------------------------------------------------
    # 1. Path Pipeline Eksplisit (Jika dijalankan berurutan)
    PATH_DIR_TRAINED: str = "/kaggle/input/training-outputs"
    PATH_DIR_PRUNED: str = "/kaggle/input/recovery-outputs"
    PATH_DIR_QUANT: str = "/kaggle/input/ptq-outputs"
    
    # 2. Path Master Fallback (Jika folder terpencar)
    MASTER_MODEL_DIR: str = "/kaggle/input/models/marrioqqqq"
    
    BASE_OUTPUT_DIR: str = "/kaggle/working/evaluation_benchmarks"

    @classmethod
    def get_output_dir(cls, arch: ModelArch, mtype: ModelType) -> str:
        """Standar 4: Mengisolasi output evaluasi berdasarkan arsitektur dan fase."""
        target_dir = os.path.join(cls.BASE_OUTPUT_DIR, arch.value, mtype.value)
        os.makedirs(target_dir, exist_ok=True)
        return target_dir

    @classmethod
    def resolve_model_path(cls, arch: ModelArch, mtype: ModelType) -> str:
        """Mencari path absolut model dengan fallback ke pencarian rekursif Master Directory."""
        
        # Kata kunci identifikasi file
        if mtype == ModelType.TRAINED:
            keywords, ext = ["phase2", "baseline", "initialized"], [".tflite", ".keras"]
            primary_dir = cls.PATH_DIR_TRAINED
        elif mtype == ModelType.PRUNED:
            keywords, ext = ["recovered", "pruned"], [".tflite", ".keras"]
            primary_dir = cls.PATH_DIR_PRUNED
        else: # QUANTIZED
            keywords, ext = ["int8", "quantized"], [".tflite"]
            primary_dir = cls.PATH_DIR_QUANT

        # Prioritas 1: Cari di folder pipeline spesifik
        if os.path.exists(primary_dir):
            for e in ext:
                for root, _, files in os.walk(primary_dir):
                    for file in files:
                        if file.endswith(e) and arch.value in file and any(k in file.lower() for k in keywords):
                            return os.path.join(root, file)

        # Prioritas 2: Cari secara rekursif di folder Master
        if os.path.exists(cls.MASTER_MODEL_DIR):
            for e in ext:
                for root, _, files in os.walk(cls.MASTER_MODEL_DIR):
                    for file in files:
                        if file.endswith(e) and arch.value in file and any(k in file.lower() for k in keywords):
                            print(f"[DEBUG] Menggunakan Fallback Master untuk {mtype.value}: {file}")
                            return os.path.join(root, file)

        raise FileNotFoundError(f"[ERROR] Model {mtype.value} untuk {arch.value} tidak ditemukan.")

    @classmethod
    def get_db_path(cls) -> str:
        """Satu database terpusat untuk membandingkan semua arsitektur dan tipe."""
        os.makedirs(cls.BASE_OUTPUT_DIR, exist_ok=True)
        return os.path.join(cls.BASE_OUTPUT_DIR, "global_evaluation_benchmark.db")

# ------------------------------------------------------------------------------
# 3. REPRODUKSIBILITAS
# ------------------------------------------------------------------------------
def set_deterministic_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    print(f"[SETUP] Deterministic Seed dikunci pada: {seed}")

set_deterministic_seed(42)

In [ ]:
# ==============================================================================
# [CELL 2] SQLITE EVALUATION LOGGER
# ==============================================================================
import sqlite3
from typing import Dict, Any

class EvalSQLiteLogger:
    """Merekam seluruh metrik klasifikasi dan performa efisiensi (Hardware)."""
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._initialize_db()

    def _initialize_db(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS evaluation_metrics (
                architecture TEXT, model_type TEXT, format TEXT,
                model_size_mb REAL, params_or_tensors INTEGER,
                realtime_latency_ms REAL, realtime_fps REAL,
                batch_latency_ms REAL, batch_throughput REAL,
                accuracy REAL, balanced_accuracy REAL, 
                macro_f1 REAL, mcc REAL, roc_auc_ovr REAL,
                PRIMARY KEY (architecture, model_type)
            )
        ''')
        conn.commit()
        conn.close()

    def log_metrics(self, data: Dict[str, Any]) -> None:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute('''
            INSERT OR REPLACE INTO evaluation_metrics 
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            data['architecture'], data['model_type'], data['format'],
            data['size_mb'], data['total_params'], 
            data['rt_latency'], data['rt_fps'], 
            data['batch_latency'], data['throughput'],
            data['accuracy'], data['balanced_acc'], 
            data['macro_f1'], data['mcc'], data['roc_auc']
        ))
        conn.commit()
        conn.close()

print("[BERHASIL] SQLite Logger Evaluasi siap digunakan.")

In [ ]:
# ==============================================================================
# [CELL 3] UNIVERSAL EVALUATION ENGINE & VISUALIZER
# ==============================================================================
import os
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef
)

plt.style.use('default')
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)

class ModelEvaluator:
    """Standar 5 & 9: Engine agnostik yang membaca TFLite maupun Keras secara transparan."""
    
    def __init__(self, config: EvalConfig, architecture: ModelArch, mtype: ModelType):
        self.cfg = config
        self.arch = architecture
        self.mtype = mtype
        self.output_dir = self.cfg.get_output_dir(self.arch, self.mtype)
        self.logger = EvalSQLiteLogger(self.cfg.get_db_path())
        
        self.model_path = self.cfg.resolve_model_path(self.arch, self.mtype)
        self.is_tflite = self.model_path.endswith('.tflite')
        
        self._load_dataset()
        
    def _load_dataset(self) -> None:
        print(f"\n[INFO] Menyiapkan Validation Dataset...")
        self.test_ds = tf.keras.utils.image_dataset_from_directory(
            self.cfg.PATH_TEST_FFB, image_size=self.cfg.IMG_SIZE, 
            batch_size=self.cfg.BATCH_SIZE, shuffle=False, label_mode='int', verbose=0
        )
        self.class_names = self.test_ds.class_names
        self.total_files = len(self.test_ds.file_paths)
        self.test_ds = self.test_ds.cache().prefetch(tf.data.AUTOTUNE)
        
        # Ekstraksi frame tunggal untuk simulasi Real-Time Camera (Batch=1)
        self.dummy_frames = [tf.expand_dims(images[0], axis=0) for images, _ in self.test_ds.unbatch().take(100)]

    # --------------------------------------------------------------------------
    # KERAS EVALUATION LOGIC
    # --------------------------------------------------------------------------
    def _evaluate_keras(self) -> dict:
        print(f"[INFO] Mengeksekusi Inferensi Format .KERAS...")
        model = tf.keras.models.load_model(self.model_path, compile=False)
        total_params = model.count_params()
        
        _ = model.predict(tf.zeros((1, 224, 224, 3)), verbose=0) # Warmup
        
        rt_latencies = []
        for frame in self.dummy_frames:
            start_rt = time.time()
            _ = model.predict_on_batch(frame)
            rt_latencies.append(time.time() - start_rt)
            
        y_true, y_pred_probs, inf_time = [], [], 0.0
        for images, labels in self.test_ds:
            y_true.extend(labels.numpy())
            start_batch = time.time()
            preds = model.predict_on_batch(images)
            inf_time += (time.time() - start_batch)
            y_pred_probs.append(preds) # Multi-class probabilities
            
        return total_params, np.mean(rt_latencies), inf_time, np.array(y_true), np.vstack(y_pred_probs)

    # --------------------------------------------------------------------------
    # TFLITE EVALUATION LOGIC
    # --------------------------------------------------------------------------
    def _evaluate_tflite(self) -> dict:
        print(f"[INFO] Mengeksekusi Inferensi Format .TFLITE...")
        interpreter = tf.lite.Interpreter(model_path=self.model_path)
        interpreter.allocate_tensors()
        i_det = interpreter.get_input_details()[0]
        o_det = interpreter.get_output_details()[0]
        
        total_tensors = sum([np.prod(t["shape"]) for t in interpreter.get_tensor_details() if len(t["shape"]) > 0])
        interpreter.resize_tensor_input(i_det['index'], [1, 224, 224, 3])
        interpreter.allocate_tensors()
        
        rt_latencies = []
        for frame in self.dummy_frames:
            img_np = frame.numpy()
            if i_det['dtype'] == np.int8: img_np = (img_np.astype(np.float32) - 128).astype(np.int8)
            
            start_rt = time.time()
            interpreter.set_tensor(i_det['index'], img_np)
            interpreter.invoke()
            _ = interpreter.get_tensor(o_det['index'])
            rt_latencies.append(time.time() - start_rt)

        y_true, y_pred_probs, inf_time = [], [], 0.0
        for images, labels in self.test_ds:
            y_true.extend(labels.numpy())
            images_np = images.numpy()
            if i_det['dtype'] == np.int8: images_np = (images_np.astype(np.float32) - 128).astype(np.int8)
            
            if list(interpreter.get_input_details()[0]['shape']) != list(images_np.shape):
                interpreter.resize_tensor_input(i_det['index'], images_np.shape)
                interpreter.allocate_tensors()
                
            interpreter.set_tensor(i_det['index'], images_np)
            start_batch = time.time()
            interpreter.invoke()
            inf_time += (time.time() - start_batch)
            
            out_data = interpreter.get_tensor(o_det['index'])
            if o_det['dtype'] != np.float32:
                scale, zp = o_det['quantization']
                out_data = (out_data.astype(np.float32) - zp) * scale
            y_pred_probs.append(out_data)
            
        return total_tensors, np.mean(rt_latencies), inf_time, np.array(y_true), np.vstack(y_pred_probs)

    # --------------------------------------------------------------------------
    # METRICS CALCULATION & PLOTTING
    # --------------------------------------------------------------------------
    def run_evaluation(self) -> None:
        print("\n" + "═"*70)
        print(f"🔬 EVALUASI MODEL: {self.arch.value} | {self.mtype.value.upper()}")
        print(f"📂 Path: {self.model_path}")
        print("═"*70)

        # 1. Ekstraksi Prediksi
        if self.is_tflite:
            params, rt_lat, inf_time, y_true, y_probs = self._evaluate_tflite()
        else:
            params, rt_lat, inf_time, y_true, y_probs = self._evaluate_keras()

        y_pred = np.argmax(y_probs, axis=1)
        size_mb = os.path.getsize(self.model_path) / (1024 * 1024)

        # 2. Kalkulasi Hardware Metrics
        rt_fps = 1.0 / rt_lat
        batch_latency_ms = (inf_time / self.total_files) * 1000
        throughput_fps = self.total_files / inf_time

        # 3. Kalkulasi Klasifikasi Metrics (Multi-Class / Macro)
        acc = accuracy_score(y_true, y_pred)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        macro_f1 = f1_score(y_true, y_pred, average='macro')
        mcc = matthews_corrcoef(y_true, y_pred)
        roc_auc = roc_auc_score(y_true, y_probs, multi_class='ovr')

        # 4. Cetak Laporan Teks
        print("\n[PERFORMANCE METRICS]")
        print(f"Accuracy      : {acc*100:.2f}%")
        print(f"Balanced Acc  : {bal_acc*100:.2f}%")
        print(f"Macro F1-Score: {macro_f1:.4f}")
        print(f"MCC           : {mcc:.4f}")
        print(f"ROC-AUC (OvR) : {roc_auc:.4f}")
        print("\n[EFFICIENCY METRICS]")
        print(f"Model Size    : {size_mb:.2f} MB")
        print(f"Complexity    : {params:,} (Params/Tensors)")
        print(f"Real-Time FPS : {rt_fps:.1f} FPS ({rt_lat*1000:.2f} ms/frame)")
        print(f"Throughput    : {throughput_fps:.1f} img/sec")
        
        print("\n" + classification_report(y_true, y_pred, target_names=self.class_names))

        # 5. Logging Database
        self.logger.log_metrics({
            'architecture': self.arch.value, 'model_type': self.mtype.value,
            'format': 'TFLite' if self.is_tflite else 'Keras', 'size_mb': size_mb,
            'total_params': params, 'rt_latency': rt_lat*1000, 'rt_fps': rt_fps,
            'batch_latency': batch_latency_ms, 'throughput': throughput_fps,
            'accuracy': acc, 'balanced_acc': bal_acc, 'macro_f1': macro_f1,
            'mcc': mcc, 'roc_auc': roc_auc
        })

        # 6. Visualisasi Terpadu
        self._generate_plots(y_true, y_pred, y_probs)

    def _generate_plots(self, y_true, y_pred, y_probs) -> None:
        print("[INFO] Mengekspor Grafik Evaluasi...")
        
        # Plot 1: Confusion Matrix
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=self.class_names, yticklabels=self.class_names)
        plt.title(f"Confusion Matrix ({self.mtype.value})", fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "confusion_matrix.png"), dpi=300)
        plt.close()

        # Plot 2: Probability Confidence Distribution
        max_probs = np.max(y_probs, axis=1)
        plt.figure(figsize=(7, 4))
        for i, cls_name in enumerate(self.class_names):
            sns.histplot(max_probs[y_true == i], bins=20, label=cls_name, kde=True, alpha=0.5)
        plt.title(f"Distribusi Konfidensi Model ({self.mtype.value})", fontweight='bold')
        plt.xlabel("Max Predicted Probability"); plt.ylabel("Jumlah Citra")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, "confidence_distribution.png"), dpi=300)
        plt.close()

print("[BERHASIL] Cell 3 (Universal Evaluator) dimuat.")

In [ ]:
# ==============================================================================
# [CELL 4] EVALUATION BATCH EXECUTOR
# ==============================================================================
import gc
import tensorflow as tf

config = EvalConfig()

# Konfigurasi matriks penelitian yang ingin dievaluasi
TARGET_ARCHITECTURES = [
    ModelArch.MOBILENET_V3,
    # ModelArch.EFFICIENTNET_B0
]

TARGET_MODEL_TYPES = [
    ModelType.TRAINED,
    ModelType.PRUNED,
    ModelType.QUANTIZED
]

print("\n" + "★"*75)
print(f"🏁 MEMULAI BENCHMARKING UNTUK {len(TARGET_ARCHITECTURES)} ARSITEKTUR LINTAS {len(TARGET_MODEL_TYPES)} FASE")
print("★"*75)

for arch in TARGET_ARCHITECTURES:
    for mtype in TARGET_MODEL_TYPES:
        try:
            # Isolasi Memori GPU Ekstrem
            tf.keras.backend.clear_session()
            gc.collect()
            set_deterministic_seed(42)
            
            evaluator = ModelEvaluator(config=config, architecture=arch, mtype=mtype)
            evaluator.run_evaluation()
            
            del evaluator
            gc.collect()
            
        except FileNotFoundError as e:
            print(f"\n[SKIP] Melompati {arch.value} - {mtype.value}: {e}")
        except Exception as e:
            print(f"\n[CRITICAL ERROR] Gagal mengevaluasi {arch.value} - {mtype.value}: {e}")

print("\n" + "★"*75)
print(f"🏆 SELURUH PROSES EVALUASI & BENCHMARKING SELESAI!")
print(f"Database Global tersimpan di: {config.get_db_path()}")
print("★"*75)

In [ ]:
1. data visualisasi kurang banyak
2. metrik tentukan apa saja
3. top-1 accuracy apakah perlu?
